<a href="https://colab.research.google.com/github/jdasam/mas1004-assignment2/blob/main/assignment2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 on Google Colab

Colab is where you train on a GPU. You do not write or change anything here.

1. On your computer, ask your agent to run something on Colab, for example
   "Train the balanced recipe on Colab with seeds 0 and 1". It writes the
   commands into `colab_runs.txt` in your repository and pushes it.
2. Here, choose Runtime, Run all.
3. When everything has run, your browser downloads `from_colab.zip`. Tell your
   agent, and it brings the results into your repository.

The first time, Colab asks you to connect your Google Drive, and the notebook
asks for the link to the starter photos (on Cyber Campus). Your Drive keeps the
code, the results and the link between sessions, in `My Drive/mas1004/assignment2`.
A run that was finished before the session ended is not run again.

## 1. Install the versions the grader uses

torch 2.6.0 and torchvision 0.21.0, the versions the grader trains Track A
recipes with. Colab comes with newer ones. This takes a minute or two.

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0 onnx==1.19.1 onnxruntime==1.23.2 onnxscript

In [ ]:
import torch, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("There is no GPU in this session. Choose Runtime, Change runtime type, T4 GPU, "
                     "Save, and then Runtime, Run all again. If Colab says no GPU is available, "
                     "try again later.")
print("GPU     ", torch.cuda.get_device_name(0))

## 2. Connect your Google Drive

A window asks you to choose your Google account and allow access. It is your
own Drive, so that your results survive when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Bring your code from GitHub

Your agent wrote the address of your repository below. The first time, this
copies your repository into `My Drive/mas1004/assignment2`. After that it makes
the code there the same as GitHub now and prints the commit, which should be
the one your agent just pushed. `data/`, `results/` and `submission/` are not
touched.

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

# Your own repository, for example "https://github.com/hong-gildong/mas1004-assignment2"
REPO = "https://github.com/YOUR-NAME/YOUR-REPOSITORY"

TEMPLATE = "https://github.com/jdasam/mas1004-assignment2"
HOME = Path("/content/drive/MyDrive/mas1004")
PROJECT = HOME / "assignment2"


def git(*args):
    done = subprocess.run(["git", *args], capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{done.stderr}")
    return done.stdout


def address(url):
    return url.strip().rstrip("/").removesuffix(".git").lower()


if "YOUR-NAME" in REPO:
    REPO = input("The address of your repository on GitHub: ").strip()
if address(REPO) == address(TEMPLATE):
    raise ValueError("That is the template. Use the address of your own repository.")

# git refuses to work in a folder it thinks belongs to another user, and folders
# in Drive can look like that.
git("config", "--global", "--add", "safe.directory", "*")

HOME.mkdir(parents=True, exist_ok=True)
old = None
if PROJECT.exists():
    try:
        here = address(git("-C", str(PROJECT), "remote", "get-url", "origin"))
    except RuntimeError:
        here = None
    if here != address(REPO):
        old = HOME / time.strftime("assignment2_old_%m%d_%H%M")
        PROJECT.rename(old)
        print(f"{PROJECT} came from {here}, so it is now {old}")

if not PROJECT.exists():
    git("clone", "-q", REPO.strip(), str(PROJECT))
    print(f"copied {REPO.strip()} into {PROJECT}")
    for name in ("data", "results", "submission"):
        if old is not None and (old / name).exists():
            shutil.move(str(old / name), str(PROJECT / name))
            print(f"moved {name}/ over from {old}")

git("-C", str(PROJECT), "fetch", "-q")
git("-C", str(PROJECT), "reset", "-q", "--hard", "@{u}")
print(git("-C", str(PROJECT), "log", "-1", "--format=code from your commit %h (%cr): %s"))
os.chdir(PROJECT)

## 4. The starter photos, on Colab's disk

About 1.7 GB, a minute or two. Reading thousands of small files from Drive is
very slow, so they go on Colab's own disk every session, and `data/starter`
points there. Photos of your own for Track B go in
`My Drive/mas1004/assignment2/data/mine/<city>/`, where they stay.

In [ ]:
link_file = Path("data/starter_link.txt")
if not link_file.exists():
    link_file.parent.mkdir(exist_ok=True)
    link_file.write_text(input("The link to the starter photos, from Cyber Campus: ").strip() + "\n")

!python src/get_data.py --into /content/data

if not Path("/content/data/starter/train").is_dir():
    link_file.unlink()
    raise SystemExit("The starter photos did not arrive; the reason is above. Run all again: "
                     "it asks for the link again.")
for name in ("starter", "sample_target"):
    link = Path("data") / name
    if link.is_symlink() or not link.exists():
        link.unlink(missing_ok=True)
        link.symlink_to(f"/content/data/{name}")

## 5. Run what your agent queued

The commands in `colab_runs.txt`, one after another. One Track A run takes
about 5 minutes. A command that already ran with the same code is skipped.

In [ ]:
!cat colab_runs.txt
!python src/colab.py run

## 6. Take the results to your computer

This packs `results/` and `submission/` into `from_colab.zip` and your browser
downloads it. Then tell your agent that `from_colab.zip` is downloaded. The
Track A models stay here: the grader trains your recipe itself.

In [ ]:
!python src/colab.py pack
from google.colab import files
files.download("from_colab.zip")